In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/ibm-hr-analytics-attrition-dataset/WA_Fn-UseC_-HR-Employee-Attrition.csv


# 🍫 IBM HR Analytics – Employee Attrition Prediction

**Career Objective:**  
As a motivated Data Science and Machine Learning learner with hands-on experience in Python, Statistics, and machine learning algorithms, I aim to apply **data-driven insights to solve real-world HR problems**. This project demonstrates my ability to perform **EDA, preprocessing, feature engineering, modeling, and evaluation** to predict employee attrition, contributing to better workforce management and retention strategies.

# Project Objective:

Predict employee attrition using HR data by applying Logistic Regression, handling class imbalance, and identifying key factors driving turnover to support data-driven HR decisions.

## Import Libraries
We import necessary libraries for **data manipulation, preprocessing, modeling, and evaluation**.

In [2]:
import os
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

os.listdir('/kaggle/input')
os.listdir('/kaggle/input/ibm-hr-analytics-attrition-dataset')

['WA_Fn-UseC_-HR-Employee-Attrition.csv']

## Load Dataset
- Load the IBM HR Analytics dataset from Kaggle input.  
- Select relevant features for modeling: `Age`, `MonthlyIncome`, `JobRole`, `YearsAtCompany`, `OverTime`, `Attrition`.

In [3]:
df= pd.read_csv('/kaggle/input/ibm-hr-analytics-attrition-dataset/WA_Fn-UseC_-HR-Employee-Attrition.csv')


df= df[
['Age','MonthlyIncome','JobRole','YearsAtCompany','OverTime','Attrition']
]

df.head()

,Age,MonthlyIncome,JobRole,YearsAtCompany,OverTime,Attrition
0,41,5993,Sales Executive,6,Yes,Yes
1,49,5130,Research Scientist,10,No,No
2,37,2090,Laboratory Technician,0,Yes,Yes
3,33,2909,Research Scientist,8,Yes,No
4,27,3468,Laboratory Technician,2,No,No


## Exploratory Data Analysis (EDA)
- Check target variable distribution (`Attrition`)  
- Check for missing values  
- Understand data types and initial statistics

In [4]:
# EDA
df['Attrition'].value_counts()

Attrition
No     1233
Yes     237
Name: count, dtype: int64

In [5]:
# Missing Value check
df.isnull().sum()

Age               0
MonthlyIncome     0
JobRole           0
YearsAtCompany    0
OverTime          0
Attrition         0
dtype: int64

# Data Preprocessing
Encode categorical variables (OverTime, Attrition)
One-hot encode JobRole
Verify transformed dataset

In [6]:
# Categorical feature encode
df['OverTime']= df['OverTime'].map({'Yes':1,'No':0})

df['Attrition']= df['Attrition'].map({'Yes':1,'No':0})

df_job= pd.get_dummies(df,columns= ['JobRole'],drop_first= True)

print('Overtime:',df['OverTime'])
print('Attrition:',df['Attrition'])
print('JobRole:',df_job)

Overtime: 0       1
1       0
2       1
3       1
4       0
       ..
1465    0
1466    0
1467    1
1468    0
1469    0
Name: OverTime, Length: 1470, dtype: int64
Attrition: 0       1
1       0
2       1
3       0
4       0
       ..
1465    0
1466    0
1467    0
1468    0
1469    0
Name: Attrition, Length: 1470, dtype: int64
JobRole:       Age  MonthlyIncome  YearsAtCompany  OverTime  Attrition  \
0      41           5993               6         1          1   
1      49           5130              10         0          0   
2      37           2090               0         1          1   
3      33           2909               8         1          0   
4      27           3468               2         0          0   
...   ...            ...             ...       ...        ...   
1465   36           2571               5         0          0   
1466   39           9991               7         0          0   
1467   27           6142               6         1          0   
1468   49    

# Separate Features and Target
X → Feature matrix
y → Target variable (Attrition)
Prepare data for training

In [7]:
#Separate features and target
x= df_job.drop('Attrition',axis=1)
y= df_job['Attrition']

# Train-Test Split
Split data into training (80%) and testing (20%) sets
Use stratified sampling to maintain target distribution

In [8]:
#Train-Test split
x_train,x_test,y_train,y_test= train_test_split(x,y,test_size=0.2,random_state= 42,stratify=y)


# Handle Class Imbalance with SMOTE
Apply SMOTE to training data to balance the minority class (Attrition=1)
Improves model performance on imbalanced datasets

In [9]:
# Smote
smote= SMOTE(random_state=42)
x_train_res,y_train_res= smote.fit_resample(x_train,y_train)


# Feature Scaling
Standardize features using StandardScaler
Ensures mean = 0, standard deviation = 1
Important for Logistic Regression performance

In [10]:
# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(x_train_res)
X_test_scaled = scaler.transform(x_test)

# Train Logistic Regression Model
Train a Logistic Regression model on scaled training data

In [11]:
# Logistic Regression Train
model= LogisticRegression(max_iter=1000)
model.fit(X_train_scaled,y_train_res)

LogisticRegression(max_iter=1000)

# Model Evaluation
Evaluate model on test data using:
Accuracy
Confusion Matrix
Classification Report (Precision, Recall, F1-score)
Assess how well the model predicts attrition

In [12]:
# Model Evaluation
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report

y_pred= model.predict(X_test_scaled)

print('Accuracy:', accuracy_score(y_test,y_pred))
print('Confusion Matrix:', confusion_matrix(y_test,y_pred))
print('Classfication Report:', classification_report(y_test,y_pred))


Accuracy: 0.7448979591836735
Confusion Matrix: [[202  45]
 [ 30  17]]
Classfication Report:               precision    recall  f1-score   support

           0       0.87      0.82      0.84       247
           1       0.27      0.36      0.31        47

    accuracy                           0.74       294
   macro avg       0.57      0.59      0.58       294
weighted avg       0.78      0.74      0.76       294



# Feature Importance
Identify which features contribute most to predicting attrition
Helps HR managers understand key attrition drivers

In [13]:
#  Feature importance 

importance= pd.Series(
    model.coef_[0],
    index= x.columns
).sort_values(ascending=False)


# Predicting New Employee Attrition
Example prediction for a new employee
Predict whether the employee is likely to leave (Churn) or stay (Not Churn)
Demonstrates practical application of the model

In [14]:
# employee attrition predict
new_employee= pd.DataFrame([{
    "Age": 29,
    "MonthlyIncome": 32000,
    "YearsAtCompany": 2,
    "OverTime": 1,
    **{col:0 for col in x.columns if col.startswith('JobRole_')}
}])

new_employee_scaled= scaler.transform(new_employee)
prediction= model.predict(new_employee_scaled)

print('Churn' if prediction[0] == 1 else 'Not Churn' )

Churn
